In [0]:
### Importando bibliotecas
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [0]:
df_cadastro_EDA_01 = spark.read.parquet(
    '/Volumes/hackathon_2025/default/source/base_dados_cadastrais/'
)   
display(df_cadastro_EDA_01)

In [0]:
# verificando o tipo dos dados
df_cadastro_EDA_01 = spark.read.parquet(
    '/Volumes/hackathon_2025/default/source/base_dados_cadastrais/') 
df_cadastro_EDA_01.dtypes

In [0]:
#listando as colunas
list(df_cadastro_EDA_01.columns)

In [0]:
# quantas linhas tem o dataframe
num_rows = df_cadastro_EDA_01.count() 
num_cols = len(df_cadastro_EDA_01.columns)
print(f"Rows: {num_rows}, Columns: {num_cols}")

In [0]:
# Calculando a idade e criando faixas etárias

from pyspark.sql import functions as F

# Corrigindo o tipo da coluna DATADENASCIMENTO para Date
# Calculando a idade
# Usa F.to_date para converter string 'dd/MM/yyyy' para Date

# Adiciona coluna de data convertida
# (alternativamente, pode-se fazer inline, mas aqui é mais claro)
df_cadastro_EDA_01 = df_cadastro_EDA_01.withColumn(
    "DATADENASCIMENTO_DATE", F.to_date(F.col("DATADENASCIMENTO"), "dd/MM/yyyy")
)
df_cadastro_EDA_01 = df_cadastro_EDA_01.withColumn(
    "IDADE", F.floor(F.datediff(F.current_date(), F.col("DATADENASCIMENTO_DATE")) / 365.25)
)

# Criando faixas etárias
df_segmentacao = df_cadastro_EDA_01.withColumn(
    "FAIXA_ETARIA",
    F.when(F.col("IDADE") < 25, "18-24")
     .when((F.col("IDADE") >= 25) & (F.col("IDADE") <= 40), "25-40")
     .when((F.col("IDADE") >= 41) & (F.col("IDADE") <= 60), "41-60")
     .otherwise("60+")
)

# Analisando Migração por Faixa Etária e Safra
df_migracao = df_segmentacao.groupBy("SAFRA", "FAIXA_ETARIA").agg(
    
    F.count("NUM_CPF").alias("volume_vendas")
)

display(df_migracao)


Databricks visualization. Run in Databricks to view.

In [0]:
display(df_cadastro_EDA_01)

In [0]:
df_cadastro_EDA_01.registerTempTable('df')

In [0]:
df_segmentacao.registerTempTable('df_segmentacao')


In [0]:
%sql
select safra,
      idade,
      count(*) as qtd
from df
where FPD is not null and flag_mig2 = "PRE"
group by safra, idade
order by safra, idade

In [0]:
%sql
select * 
from df
where (idade is null or idade < 18) and
 FPD is not null and flag_mig2 = "PRE"

In [0]:
%sql
select safra,
      avg(idade)
      
from df
where FPD is not null and flag_mig2 = "PRE" and FPD = 1

group by safra
order by safra

In [0]:
%sql
select * from df_segmentacao

In [0]:
%sql
SELECT SAFRA,
       FAIXA_ETARIA,
       ROUND(SUM(FPD)/COUNT(*),2) AS PCT_FPD,
       COUNT(*) AS QTD
FROM df_segmentacao
WHERE FPD IS NOT NULL  AND flag_mig2 = 'PRE'
GROUP BY SAFRA, FAIXA_ETARIA
ORDER BY SAFRA, faixa_etaria

Databricks visualization. Run in Databricks to view.